# Weather forecasting: direct multi-step LSTM (all districts)

Forecasts all 11 daily weather variables **3 days ahead** for each of the 5 districts, one model per district.

**Design** (worked out in the Latur prototype):
- **Direct multi-step head.** One LSTM encodes the 45-day lookback window, and an MLP outputs every forecast day at once. The earlier repeat-vector decoder flattened the forecasts.
- **Known future calendar.** The model also sees the sin/cos day-of-year of each day it forecasts.
- **log1p on rainfall and wind.** Both are heavily skewed. The loss is Huber, which is robust to rare rainfall spikes.
- **Chronological split.** Train ≤ 2018, validation 2019–2021, test 2022 onward. Scalers are fit on the training split only.

**Input:** `data/weather/clean/` (run `../preprocessing.ipynb` first).
**Output:** weights go to `models/<district>_weather_lstm_pytorch.pt`. Scalers, `config.json`, metrics and plots go to `outputs/lstm/<District>/`.

Set `RUN_TRAINING = True` to retrain (about 10-15 min per district on CPU). With `False`, the notebook loads the saved models and re-evaluates them.

## 1. Setup & configuration

In [ ]:
import copy
import json
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "codes").is_dir() and (p / "data").is_dir())
CLEAN_DIR = ROOT / "data" / "weather" / "clean"
MODELS_DIR = ROOT / "models"
OUTPUT_DIR = ROOT / "outputs" / "lstm"

# ---- Run configuration ----
DISTRICTS = ["Latur", "Beed", "Osmanabad", "Solapur", "Ahmednagar"]
RUN_TRAINING = False   # False = load saved models and evaluate; True = retrain and overwrite
SHOW_PLOTS = True      # show figures inline (they are always saved)

# ---- Data split ----
TRAIN_END = "2018-12-31"
VAL_END = "2021-12-31"

# ---- Model / training hyperparameters ----
HORIZON = 3
LOOKBACK = 45
LSTM_UNITS = 128
NUM_LSTM_LAYERS = 1
DENSE_UNITS = 384
DROPOUT = 0.25
LEARNING_RATE = 5e-4
BATCH_SIZE = 16
MAX_EPOCHS = 80
PATIENCE = 12
SEED = 42
LOG_COLS = ["Rainfall_mm", "Wind_Speed_mps"]

print("device:", DEVICE, "| repo root:", ROOT)

## 2. Model & helpers

In [ ]:
class DirectLSTM(nn.Module):
    def __init__(self, n_input, n_target, horizon, lstm_units, num_layers, dense_units, dropout):
        super().__init__()
        self.lstm = nn.LSTM(n_input, lstm_units, num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.horizon = horizon
        self.n_target = n_target
        self.drop_lstm = nn.Dropout(dropout)
        self.fc1 = nn.Linear(lstm_units + horizon * 2, dense_units)
        self.drop1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(dense_units, dense_units // 2)
        self.drop2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(dense_units // 2, horizon * n_target)
        self.act = nn.ReLU()

    def forward(self, seq, cal):
        _, (h, _) = self.lstm(seq)
        enc = self.drop_lstm(h[-1])                 # final layer's hidden state
        cal_flat = cal.reshape(cal.size(0), -1)     # (batch, horizon*2)
        x = torch.cat([enc, cal_flat], dim=1)
        x = self.drop1(self.act(self.fc1(x)))
        x = self.drop2(self.act(self.fc2(x)))
        x = self.fc3(x)
        return x.view(-1, self.horizon, self.n_target)


def calendar_feats(index):
    """sin/cos of day-of-year: always known in advance for a future date."""
    doy = np.asarray(index.dayofyear)
    leap = np.asarray(index.is_leap_year)
    frac = doy / np.where(leap, 366, 365)
    return np.stack([np.sin(2 * np.pi * frac), np.cos(2 * np.pi * frac)], axis=1)


def make_windows(in_arr, out_arr, cal_arr, lookback, horizon):
    Xs, Ys, Cs = [], [], []
    for i in range(len(in_arr) - lookback - horizon + 1):
        Xs.append(in_arr[i:i + lookback])
        Ys.append(out_arr[i + lookback:i + lookback + horizon])
        Cs.append(cal_arr[i + lookback:i + lookback + horizon])
    return (np.array(Xs, dtype=np.float32), np.array(Ys, dtype=np.float32), np.array(Cs, dtype=np.float32))


def make_windows_with_context(in_ctx, out_ctx, cal_ctx, in_arr, out_arr, cal_arr, lookback, horizon):
    """Prepend the tail of the previous split so a split's first `lookback` days aren't lost."""
    full_in = np.concatenate([in_ctx[-lookback:], in_arr], axis=0)
    full_out = np.concatenate([out_ctx[-lookback:], out_arr], axis=0)
    full_cal = np.concatenate([cal_ctx[-lookback:], cal_arr], axis=0)
    return make_windows(full_in, full_out, full_cal, lookback, horizon)


def inverse_target(scaled_seq, scaler, log_cols, feature_names):
    """Undo MinMax scaling, then log1p on log_cols."""
    n, h, f = scaled_seq.shape
    inv = scaler.inverse_transform(scaled_seq.reshape(-1, f)).reshape(n, h, f)
    for c in log_cols:
        j = feature_names.index(c)
        inv[:, :, j] = np.expm1(inv[:, :, j])
    return inv


def model_path(district):
    return MODELS_DIR / f"{district.lower()}_weather_lstm_pytorch.pt"

## 3. Data preparation

Load the cleaned CSV, apply `log1p` to the skewed columns, add causal rolling features (past values only),
then split chronologically into train / validation / test.

In [ ]:
ENGINEERED = ["Rain_roll7", "Rain_roll14", "Rain_roll30", "Humid_roll7",
              "Solar_roll7", "Wind_roll7", "Wind50_roll7", "Cloud_roll7"]


def prepare_data(district):
    df = pd.read_csv(CLEAN_DIR / f"{district.lower()}_weather.csv", parse_dates=["DATE"]).set_index("DATE")
    target_features = list(df.columns)

    df_model = df.copy()
    for c in LOG_COLS:
        df_model[c] = np.log1p(df_model[c])
    df_model["Rain_roll7"] = df["Rainfall_mm"].rolling(7, min_periods=1).sum()
    df_model["Rain_roll14"] = df["Rainfall_mm"].rolling(14, min_periods=1).sum()
    df_model["Rain_roll30"] = df["Rainfall_mm"].rolling(30, min_periods=1).sum()
    df_model["Humid_roll7"] = df["Relative_Humidity_percent"].rolling(7, min_periods=1).mean()
    df_model["Solar_roll7"] = df["Solar_Radiation"].rolling(7, min_periods=1).mean()
    df_model["Wind_roll7"] = df["Wind_Speed_mps"].rolling(7, min_periods=1).mean()
    df_model["Wind50_roll7"] = df["Wind_Speed_50m_mps"].rolling(7, min_periods=1).mean()
    df_model["Cloud_roll7"] = df["Cloud_Amount_percent"].rolling(7, min_periods=1).mean()

    train_df = df_model.loc[:TRAIN_END]
    val_df = df_model.loc[pd.Timestamp(TRAIN_END) + pd.Timedelta(days=1):VAL_END]
    test_df = df_model.loc[pd.Timestamp(VAL_END) + pd.Timedelta(days=1):]
    return {"df": df, "train": train_df, "val": val_df, "test": test_df,
            "targets": target_features, "inputs": target_features + ENGINEERED}

## 4. Training

Huber loss and Adam, with the learning rate halved when validation loss plateaus.
Early stopping restores the best validation weights.

In [ ]:
def train_model(district, data):
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    targets, inputs = data["targets"], data["inputs"]
    train_df, val_df = data["train"], data["val"]

    in_scaler, out_scaler = MinMaxScaler(), MinMaxScaler()
    train_in = in_scaler.fit_transform(train_df[inputs])
    val_in = in_scaler.transform(val_df[inputs])
    train_out = out_scaler.fit_transform(train_df[targets])
    val_out = out_scaler.transform(val_df[targets])
    cal_train, cal_val = calendar_feats(train_df.index), calendar_feats(val_df.index)

    Xtr, Ytr, Ctr = make_windows(train_in, train_out, cal_train, LOOKBACK, HORIZON)
    Xval, Yval, Cval = make_windows_with_context(train_in, train_out, cal_train,
                                                 val_in, val_out, cal_val, LOOKBACK, HORIZON)
    loader = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(Ctr), torch.from_numpy(Ytr)),
                        batch_size=BATCH_SIZE, shuffle=True)
    Xval_t, Cval_t, Yval_t = (torch.from_numpy(a).to(DEVICE) for a in (Xval, Cval, Yval))

    model = DirectLSTM(len(inputs), len(targets), HORIZON, LSTM_UNITS, NUM_LSTM_LAYERS,
                       DENSE_UNITS, DROPOUT).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=4)
    loss_fn = nn.HuberLoss(delta=1.0)

    history = {"loss": [], "val_loss": []}
    best_val, best_state, bad_epochs = float("inf"), None, 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        running, n_batches = 0.0, 0
        for xb, cb, yb in loader:
            xb, cb, yb = xb.to(DEVICE), cb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb, cb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item()
            n_batches += 1
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xval_t, Cval_t), Yval_t).item()
        scheduler.step(val_loss)
        history["loss"].append(running / n_batches)
        history["val_loss"].append(val_loss)

        if val_loss < best_val - 1e-5:
            best_val, best_state, bad_epochs = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break
    model.load_state_dict(best_state)
    print(f"[{district}] {len(history['loss'])} epochs, best val loss {best_val:.5f}")
    return model, in_scaler, out_scaler, history


def save_model(district, data, model, in_scaler, out_scaler, blended_r2):
    out_dir = OUTPUT_DIR / district
    out_dir.mkdir(parents=True, exist_ok=True)
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model_state_dict": model.state_dict(),
        "model_config": {"n_input": len(data["inputs"]), "n_target": len(data["targets"]), "horizon": HORIZON,
                         "lstm_units": LSTM_UNITS, "num_layers": NUM_LSTM_LAYERS,
                         "dense_units": DENSE_UNITS, "dropout": DROPOUT},
    }, model_path(district))
    joblib.dump(in_scaler, out_dir / "in_scaler.joblib")
    joblib.dump(out_scaler, out_dir / "out_scaler.joblib")
    config = {
        "district": district, "target_features": data["targets"], "input_features": data["inputs"],
        "log_transformed_columns": LOG_COLS, "lookback": LOOKBACK, "horizon": HORIZON,
        "lstm_units": LSTM_UNITS, "num_layers": NUM_LSTM_LAYERS, "dense_units": DENSE_UNITS,
        "dropout": DROPOUT, "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE,
        "train_end": TRAIN_END, "val_end": VAL_END, "blended_test_r2": blended_r2,
        "trained_through_date": str(data["df"].index.max().date()),
    }
    (out_dir / "config.json").write_text(json.dumps(config, indent=2))


def load_model(district):
    ckpt = torch.load(model_path(district), map_location=DEVICE, weights_only=False)
    mc = ckpt["model_config"]
    model = DirectLSTM(mc["n_input"], mc["n_target"], mc["horizon"], mc["lstm_units"],
                       mc["num_layers"], mc["dense_units"], mc["dropout"]).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    out_dir = OUTPUT_DIR / district
    return model, joblib.load(out_dir / "in_scaler.joblib"), joblib.load(out_dir / "out_scaler.joblib")

## 5. Evaluation

Predictions on the test period are converted back to original units. RMSE, MAE and R² are computed per variable,
pooled over all three forecast days. The **blended R²** is the mean of the per-variable R² values.

In [ ]:
def evaluate(data, model, in_scaler, out_scaler):
    targets, inputs = data["targets"], data["inputs"]
    trainval = pd.concat([data["train"], data["val"]])
    Xtest, Ytest, Ctest = make_windows_with_context(
        in_scaler.transform(trainval[inputs]), out_scaler.transform(trainval[targets]),
        calendar_feats(trainval.index),
        in_scaler.transform(data["test"][inputs]), out_scaler.transform(data["test"][targets]),
        calendar_feats(data["test"].index), LOOKBACK, HORIZON)
    model.eval()
    with torch.no_grad():
        pred_scaled = model(torch.from_numpy(Xtest).to(DEVICE), torch.from_numpy(Ctest).to(DEVICE)).cpu().numpy()
    y_true = inverse_target(Ytest, out_scaler, LOG_COLS, targets)
    y_pred = inverse_target(pred_scaled, out_scaler, LOG_COLS, targets)

    rows = []
    for j, feat in enumerate(targets):
        a, p = y_true[:, :, j].reshape(-1), y_pred[:, :, j].reshape(-1)
        rows.append({"feature": feat, "RMSE": np.sqrt(mean_squared_error(a, p)),
                     "MAE": mean_absolute_error(a, p), "R2": r2_score(a, p)})
    metrics = pd.DataFrame(rows).set_index("feature").round(3)
    test_dates = trainval.index[-LOOKBACK:].append(data["test"].index)
    return metrics, float(metrics["R2"].mean()), y_true, y_pred, test_dates

## 6. Plots

In [ ]:
def finish(fig, district, name):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / district / name, dpi=110, bbox_inches="tight")
    plt.show() if SHOW_PLOTS else plt.close(fig)


def plot_loss(district, history):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(history["loss"], label="train loss")
    ax.plot(history["val_loss"], label="validation loss")
    ax.set_xlabel("epoch")
    ax.set_ylabel("Huber loss (scaled)")
    ax.set_title(f"{district} — training curves")
    ax.legend()
    finish(fig, district, "loss_curve.png")


def plot_evaluation(district, targets, metrics, blended_r2, y_true, y_pred, test_dates):
    # Per-feature R2
    fig, ax = plt.subplots(figsize=(9, 4.5))
    colors = ["#4c72b0" if v >= 0.8 else ("#dd8452" if v >= 0.5 else "#c44e52") for v in metrics["R2"]]
    ax.bar(metrics.index, metrics["R2"], color=colors)
    ax.axhline(0.8, color="black", linestyle="--", lw=1, label="R² = 0.8 reference")
    ax.axhline(blended_r2, color="#4c72b0", linestyle=":", lw=1.5, label=f"blended R² = {blended_r2:.3f}")
    ax.set_ylabel("R² (test set, pooled over all forecast days)")
    ax.set_title(f"{district} — per-feature R² ({HORIZON}-day-ahead forecast)")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
    finish(fig, district, "r2_by_feature.png")

    # Parity plots (actual vs predicted)
    fig, axes = plt.subplots(3, 4, figsize=(16, 11))
    for ax in axes.flat[len(targets):]:
        ax.axis("off")
    for ax, feat in zip(axes.flat, targets):
        j = targets.index(feat)
        a, p = y_true[:, :, j].reshape(-1), y_pred[:, :, j].reshape(-1)
        ax.scatter(a, p, s=4, alpha=0.15, color="#4c72b0")
        lims = [min(a.min(), p.min()), max(a.max(), p.max())]
        ax.plot(lims, lims, color="black", lw=1, linestyle="--")
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_title(f"{feat}\nR² = {metrics.loc[feat, 'R2']:.3f}", fontsize=10)
        ax.set_xlabel("actual")
        ax.set_ylabel("predicted")
    fig.suptitle(f"{district} — parity plots", y=1.01)
    finish(fig, district, "parity_plots.png")

    # Sample 3-day forecasts at three points in the test period
    n = len(y_true)
    positions = [n // 5, n // 2, 4 * n // 5]
    plot_features = ["Rainfall_mm", "Temperature_Max_C", "Relative_Humidity_percent"]
    fig, axes = plt.subplots(len(plot_features), len(positions), figsize=(14, 9), sharey="row")
    for col, pos in enumerate(positions):
        start = test_dates[pos + LOOKBACK]
        dates = pd.date_range(start, periods=HORIZON)
        for row, feat in enumerate(plot_features):
            j = targets.index(feat)
            ax = axes[row, col]
            ax.plot(dates, y_true[pos, :, j], marker="o", color="black", label="Actual")
            ax.plot(dates, y_pred[pos, :, j], marker="o", color="#c44e52", label="LSTM forecast")
            ax.set_xticks(dates)
            ax.set_xticklabels([d.strftime("%b %d") for d in dates])
            if row == 0:
                ax.set_title(f"Forecast from {start.date()}")
            if col == 0:
                ax.set_ylabel(feat)
            if row == 0 and col == 0:
                ax.legend(fontsize=8)
    fig.suptitle(f"{district} — sample forecasts", y=1.01)
    finish(fig, district, "sample_forecasts.png")

## 7. Run for all districts

In [ ]:
summary = {}
for district in DISTRICTS:
    print(f"===== {district} =====")
    data = prepare_data(district)
    (OUTPUT_DIR / district).mkdir(parents=True, exist_ok=True)

    if RUN_TRAINING:
        model, in_scaler, out_scaler, history = train_model(district, data)
        plot_loss(district, history)
    else:
        model, in_scaler, out_scaler = load_model(district)

    metrics, blended_r2, y_true, y_pred, test_dates = evaluate(data, model, in_scaler, out_scaler)
    metrics.to_csv(OUTPUT_DIR / district / "test_metrics_by_feature.csv")
    if RUN_TRAINING:
        save_model(district, data, model, in_scaler, out_scaler, blended_r2)
    plot_evaluation(district, data["targets"], metrics, blended_r2, y_true, y_pred, test_dates)

    summary[district] = metrics["R2"].rename(district)
    print(f"[{district}] blended test R² = {blended_r2:.4f}")

In [ ]:
# Per-variable test R² for every district (last row = blended R²)
r2_table = pd.DataFrame(summary)
r2_table.loc["BLENDED"] = r2_table.mean()
r2_table.round(3)